In [2]:
%load_ext autoreload
%autoreload 2
import numpy as np
import pandas as pd
import mne
from pydeconv.utils import *
from pydeconv.pydeconv_sims_v2 import *
from pydeconv import *
from sklearn.model_selection import KFold, GridSearchCV
from sklearn.metrics import mean_squared_error
%matplotlib qt 


# raw     = mne.io.read_raw_eeglab(data_path + "629959_analysis.set", preload=True)

# # Initialize the model

# rERP_model = PyDeconv(settings = settings , features = features, eeg = raw)
# X_design = rERP_model.create_matrix()

/home/eiff/Documents/facu/tesis/PyDeconv/.venv/lib/python3.11/site-packages/ipykernel/eventloops.py:158: UserWarning: Creating legend with loc="best" can be slow with large amounts of data.
  el.exec() if hasattr(el, "exec") else el.exec_()
/home/eiff/Documents/facu/tesis/PyDeconv/.venv/lib/python3.11/site-packages/ipykernel/eventloops.py:158: UserWarning: Creating legend with loc="best" can be slow with large amounts of data.
  el.exec() if hasattr(el, "exec") else el.exec_()
/home/eiff/Documents/facu/tesis/PyDeconv/.venv/lib/python3.11/site-packages/ipykernel/eventloops.py:158: UserWarning: Creating legend with loc="best" can be slow with large amounts of data.
  el.exec() if hasattr(el, "exec") else el.exec_()


In [40]:
# --- Build a compound ERP kernel ---
kernel = CompoundKernel(sfreq=500)
kernel.add(peak_latency=0.1,  amplitude=0.10, width=0.03, shape='gaussian', label='P1')
kernel.add(peak_latency=0.17, amplitude=-0.07, width=0.04, shape='gaussian', label='N170')
kernel.add(peak_latency=0.25, amplitude=0.04, width=0.05, shape='hanning',  label='P2')
kernel.plot()

# --- Create random event onsets ---
sfreq = 500
duration = 6000  # seconds
n_events = 100
latencies = np.sort(np.random.choice(np.arange(500, int(duration * sfreq) - 500), n_events, replace=False))
events = pd.DataFrame({'latency': latencies, 'type': 'stimulus'})

# --- Simulate ---
sim = EEGSimulator(sfreq=sfreq, duration=duration)
sim.add_kernel('stimulus', kernel)
sim.set_events(events)
sim.simulate()
sim.add_noise(colour='brown', scale=0.3)
sim.plot()

{'kwargs', '_'}
{'kwargs', '_'}
{'kwargs', '_'}


In [ ]:
# UnfoldSim notes
# unfoldsim first creates a table of *all* events on a trial. It is balanced in the sense that it's a cross product of all possible values
# You can however offset the events as you want. Then, to actually simulate the data you must place the events at a specific latency point.
# This is performed by onset_types. Each time you sample it you get the distance in samples from the next event. You can also conditionally change the distribution based on the dataframe.

# So, what we can do is:
# 1. Like we're doing, create a table of events on a trial. This is already done using TrialVariable and TrialStructure.
# 2. Then, we can have this distribution of event offsets, with conditional distribution. We can have a lambda that has access to the dataframe.
#   - Question: how do we control correlated variables? e.g. maybe we always want a fixation, saccade, fixation, sacade, ... pattern
#       - This is done by the generator function.
# 3. Then, we need to define the compound kernel like in the block above. Again, we might need access to the dataframe columns. If we have a few amount of kernels, we could generate each waveform for each table combination. Then, we convolve each one separately (this could be paralellized)


# Create sample TrialStructure
# Create a sample TrialStructure using the TrialVariable class

# Define trial variable
trial_result = TrialVariable(
    name='correct',
    generator=lambda n, _rng : np.random.choice([0, 1], size=n),
    static_accross_trial=True
)
trial_mss = TrialVariable(
    name='MSS',
    generator=lambda n, _rng : np.random.choice([1, 2, 3, 4], size=n),
    static_accross_trial=True
)
stim_type = TrialVariable(
    name='stimulus_type',
    generator=lambda n, rng: rng.choice(['fixation', 'saccade'], size=n)
)
rt_var = TrialVariable(
    name='sacc_amplitude',
    generator=lambda n, rng: rng.uniform(low=0.0, high=1.0, size=n)
)

# Specify trial structure configuration
trial_vars = [trial_result, trial_mss, stim_type, rt_var]

# Initialize TrialStructure with a number of trials (e.g., 100)
trial_struct = TrialStructure(sfreq=sfreq, variables=trial_vars)

# Generate the events DataFrame
events_df = trial_struct.generate_events_df(n_events=n_events)

# Display the resulting DataFrame
print(events_df.head())

# Add latencies to the events dataframe
events_df = assign_event_latencies(events_df, sampler=build_uniform_isi_sampler(width=50, offset=20))

# Display the resulting DataFrame
print(events_df.head())


#sim = EEGSimulator(sfreq=sfreq, duration=duration)
#sim.add_kernel('stimulus', kernel)
#sim.set_events(events_df)
#sim.simulate()
#sim.add_noise(colour='brown', scale=0.3)
#sim.plot()

modif = lambda sacc_amplitude, correct: sacc_amplitude * correct 


import inspect
modifier_signature = inspect.signature(modif)
modifier_parameters = modifier_signature.parameters.copy()
old_kwargs = modifier_parameters.get('kwargs', None)
modifier_parameters['kwargs'] = inspect.Parameter('kwargs', inspect.Parameter.VAR_KEYWORD)
kwargs_modified = modifier_signature.replace(parameters=modifier_parameters.values()).parameters['kwargs']
modif.__signature__ = modifier_signature.replace(parameters=modifier_parameters.values())

if old_kwargs != kwargs_modified:
    print("different")

modif(**events_df.iloc[0])

   correct  MSS stimulus_type  sacc_amplitude
0        1    2       saccade        0.077748
1        1    2      fixation        0.642424
2        1    2      fixation        0.618074
3        1    2       saccade        0.420711
4        1    2       saccade        0.408703
   correct  MSS stimulus_type  sacc_amplitude  latency
0        1    2       saccade        0.077748       58
1        1    2      fixation        0.642424      118
2        1    2      fixation        0.618074      187
3        1    2       saccade        0.420711      257
4        1    2       saccade        0.408703      292


KeyError: 'kwargs'